In [2]:
import numpy as np
import matplotlib.pyplot as plt

class Detector:
    def __init__(self, R, r, N):
        self.R = R
        self.r = r
        self.N = N
        self.positions = self._place_pmt()

    def _place_pmt(self):
        # Используем алгоритм Фибоначчи для равномерного распределения по сфере
        points = []
        phi = np.pi * (3. - np.sqrt(5.))  # золотое сечение
        for i in range(self.N):
            y = 1 - (i / float(self.N - 1)) * 2  # от 1 до -1
            radius = np.sqrt(1 - y * y)
            theta = phi * i
            x = np.cos(theta) * radius
            z = np.sin(theta) * radius
            points.append([x, y, z])
        
        points = np.array(points)
        # Нормализуем и умножаем на радиус R
        points = points / np.linalg.norm(points, axis=1)[:, np.newaxis] * self.R
        
        # Проверка на пересечение катодов (ФЭУ не должны перекрываться)
        for i in range(len(points)):
            for j in range(i + 1, len(points)):
                dist = np.linalg.norm(points[i] - points[j])
                if dist < 2 * self.r:
                    print(f"Внимание: ФЭУ {i} и {j} пересекаются (расстояние {dist:.4f} < {2*self.r})!")
        
        return points


class Source:
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_photons(self, E):
        N = int(E * self.Y)
        phi = np.random.uniform(0, 2 * np.pi, N)
        cos_theta = np.random.uniform(-1, 1, N)
        sin_theta = np.sqrt(1 - cos_theta**2)
        x = sin_theta * np.cos(phi)
        y = sin_theta * np.sin(phi)
        z = cos_theta
        return np.column_stack((x, y, z))


class PMT:
    def __init__(self, center, r, a):
        self.center = np.array(center)  # Координаты центра
        self.r = r                     
        self.a = a                     

    def detect_photon(self, photon_point):
        dist = np.linalg.norm(photon_point - self.center)
        if dist <= self.r:
            return np.random.poisson(self.a)
        else:
            return 0  # не в сфере


R = 2.0
r = 0.1
N_pmt = 40
Y = 10000
a = 0.5
E = 1.0

# ИСПРАВЛЕНИЕ 1: координаты источника (по условию — в центре детектора)
source_pos = np.array([0.0, 0.0, 0.0])

det = Detector(R, r, N_pmt)
source = Source(Y)

pmts = [PMT(center=pos, r=r, a=a) for pos in det.positions]

directions = source.generate_photons(E)
num_photons = len(directions)
print(f"Сгенерировано фотонов: {num_photons}")

total_electrons = 0

for dir_vec in directions:
    # ИСПРАВЛЕНИЕ 1: находим точку пересечения луча s + t*n со сферой |x| = R
    # Решаем |source_pos + t*dir_vec|^2 = R^2, берем положительный корень t
    b = np.dot(source_pos, dir_vec)
    c = np.dot(source_pos, source_pos) - R**2
    disc = b**2 - c
    if disc < 0:
        continue  # луч не пересекает сферу
    t = -b + np.sqrt(disc)  # положительный корень
    if t <= 0:
        continue
    hit_point = source_pos + t * dir_vec

    for pmt in pmts:
        electrons = pmt.detect_photon(hit_point)
        total_electrons += electrons

print(f"Полный собранный заряд Q (симуляция): {total_electrons}")

Q_expected = a * (E * Y) * (N_pmt * r**2 / R**2)

print(f"Ожидаемый заряд Q (теория): {Q_expected:.1f}")
print(f"Отклонение: {abs(total_electrons - Q_expected) / Q_expected * 100:.1f}%")

Сгенерировано фотонов: 10000
Полный собранный заряд Q (симуляция): 131
Ожидаемый заряд Q (теория): 500.0
Отклонение: 73.8%
